# Ноутбук для семинара `Трекинг экспериментов`

Сегодня мы:
- рассмотрим пайплайн подготовки данных и обучения модели классификации доходов
- создадим эксперимент в MLflow и настроим логирование:
    - параметров
    - метрик
    - моделей
    - артефактов
- проведём серию запусков с изменением параметров и сравним результаты в UI MLflow

## Импорты

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder, TargetEncoder

import mlflow
import mlflow.sklearn

/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
mlflow.set_tracking_uri("http://158.160.242.172:5000/")

EXPERIMENT_NAME = "homework-dvsobolev.ext"
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/35', creation_time=1778253455992, experiment_id='35', last_update_time=1778253455992, lifecycle_stage='active', name='homework-dvsobolev.ext', tags={}>

In [4]:
DATASET_NAME = "scikit-learn/adult-census-income"
TEST_SIZE = 0.3
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

## Загрузка и подготовка данных

In [5]:
dataset = load_dataset(DATASET_NAME)
df = dataset["train"].to_pandas()

In [6]:
columns = [
    "age", "workclass", "education.num", "occupation", 
    "marital.status", "capital.gain", "hours.per.week", 
    "sex", "race"
]
target_column = "income"

X = df[columns]
y = df[target_column]

Описание признаков:

- `age` &mdash; возраст человека
- `workclass` &mdash; тип занятости
- `fnlwgt` &mdash; вес наблюдения в данных переписи населения США (сколько реальных людей в популяции «представляет» эта строка)
- `education` &mdash; образование
- `education.num` &mdash; уровень образования в виде числа
- `marital.status` &mdash; семейное положение
- `occupation` &mdash; профессия / род деятельности
- `relationship` &mdash; роль человека в семье
- `race` &mdash; расовая группа
- `sex` &mdash; пол человека (`Male` / `Female`)
- `capital.gain` &mdash; доход от капитала (прибыль от продажи активов)
- `capital.loss` &mdash; убытки от капитала
- `hours.per.week` &mdash; количество рабочих часов в неделю
- `native.country` &mdash; страна происхождения

Описание таргета:

- `income` &mdash; бинарно, получает человек больше 50k $ в год или нет

In [7]:
y_transformed = (y == ">50K").astype(int)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y_transformed, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Train shape: (22792, 9), Test shape: (9769, 9)


## Создание пайплайна и обучение

In [9]:
cat_features = ["workclass", "occupation", "marital.status", "sex", "race"]
num_features = list(set(columns) - set(cat_features))

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", TargetEncoder(random_state=RANDOM_STATE), cat_features),
        ("num", StandardScaler(), num_features),
    ]
)

In [11]:
model_params = dict(
    penalty="l2", 
    C=1.0, 
    solver="newton-cg", 
    max_iter=100, 
    random_state=RANDOM_STATE
)
model = LogisticRegression(**model_params)

In [12]:
pipeline = Pipeline([
    ("preprocess", preprocessor), 
    ("model", model)
])

pipeline.fit(X_train, y_train)

/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

## Оценка качества и логирование в MLflow

In [13]:
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = np.where(y_proba >= 0.5, 1, 0)

In [14]:
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

In [15]:
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

accuracy: 0.8426
f1: 0.6269
roc_auc: 0.8980


In [53]:
with mlflow.start_run(run_name="baseline_logistic_regression"):
    mlflow.log_params(model_params)
    mlflow.log_params({
        "cat_features": cat_features,
        "num_features": num_features,
        "preprocessing": type(pipeline.named_steps["preprocess"].transformers[0][1]).__name__,
    })
    
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(pipeline, artifact_path="model")

2026/05/08 18:26:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/08 18:26:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run baseline_logistic_regression at: http://158.160.242.172:5000/#/experiments/35/runs/1d440773c4be417dbb38433932011e89
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


# Домашка - 20 разных экспериментов

In [26]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    average_precision_score, 
    classification_report, 
    ConfusionMatrixDisplay
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature
import itertools

In [ ]:
def run_experiment(run_name, hypothesis, model, preprocessor, X_train_exp, y_train_exp, X_test_exp, y_test_exp, cat_feat, num_feat):
    with mlflow.start_run(run_name=run_name):
        pipeline = Pipeline([
            ("preprocess", preprocessor), 
            ("model", model)
        ])
        pipeline.fit(X_train_exp, y_train_exp)
        
        y_proba = pipeline.predict_proba(X_test_exp)[:, 1]
        y_pred = np.where(y_proba >= 0.5, 1, 0)
        
        metrics = {
            "accuracy": accuracy_score(y_test_exp, y_pred),
            "precision": precision_score(y_test_exp, y_pred),
            "recall": recall_score(y_test_exp, y_pred),
            "f1": f1_score(y_test_exp, y_pred),
            "roc_auc": roc_auc_score(y_test_exp, y_proba),
            "pr_auc": average_precision_score(y_test_exp, y_proba)
        }

        mlflow.log_metrics(metrics)
        mlflow.log_params(model.get_params())
        mlflow.log_params({
            "cat_features": cat_feat, 
            "num_features": num_feat,
            "train_size": len(X_train_exp),
            "preprocessing": type(preprocessor.transformers[0][1]).__name__,
            "model_type": type(model).__name__
        })
        
        mlflow.set_tag("mlflow.note.content", hypothesis)
        
        fig, ax = plt.subplots(figsize=(6, 4))
        ConfusionMatrixDisplay.from_predictions(y_test_exp, y_pred, ax=ax, cmap="Blues")
        plt.title("Confusion Matrix")
        mlflow.log_figure(fig, "artifacts/confusion_matrix.png")
        plt.close(fig)
        
        report = classification_report(y_test_exp, y_pred, output_dict=True)
        mlflow.log_dict(report, "artifacts/classification_report.json")
        
        model_name = f"AdultIncome_{type(model).__name__}"
        signature = infer_signature(X_train_exp, pipeline.predict(X_train_exp))
        mlflow.sklearn.log_model(
            sk_model=pipeline, 
            artifact_path="model", 
            registered_model_name=model_name,
            signature=signature,
            input_example=X_train_exp.iloc[:3]
        )
        
        print(f"[{run_name}] ROC-AUC: {metrics['roc_auc']:.4f}")
        return metrics['roc_auc']

## Разные энкодеры

In [ ]:
base_model = LogisticRegression(
    C=1.0, 
    solver="newton-cg", 
    max_iter=100, 
    random_state=RANDOM_STATE
)

encoders = {
    "TargetEncoder": TargetEncoder(random_state=RANDOM_STATE),
    "OrdinalEncoder": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    "OneHotEncoder": OneHotEncoder(handle_unknown="ignore", sparse_output=False)
}

for enc_name, encoder in encoders.items():
    run_name = f"enc_{enc_name}"
    hypothesis = f"Сравнение энкодеров. {enc_name}"

    preprocessor_exp = ColumnTransformer(
        transformers=[
            ("cat", encoder, cat_features),
            ("num", StandardScaler(), num_features),
        ]
    )
    
    run_experiment(
        run_name=run_name,
        hypothesis=hypothesis,
        model=base_model,
        preprocessor=preprocessor_exp,
        X_train_exp=X_train,
        y_train_exp=y_train,
        X_test_exp=X_test,
        y_test_exp=y_test,
        cat_feat=cat_features,
        num_feat=num_features
    )

/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:45:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[enc_TargetEncoder] ROC-AUC: 0.8980
🏃 View run enc_TargetEncoder at: http://158.160.242.172:5000/#/experiments/35/runs/ddf8619d6de2478ca2d998c1a08c5832
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:46:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[enc_OrdinalEncoder] ROC-AUC: 0.8509
🏃 View run enc_OrdinalEncoder at: http://158.160.242.172:5000/#/experiments/35/runs/3371d9aaa5f2475683d347ca5105165c
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:46:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[enc_OneHotEncoder] ROC-AUC: 0.9014
🏃 View run enc_OneHotEncoder at: http://158.160.242.172:5000/#/experiments/35/runs/e6679d26679d4cc5a5c2b6b6395b2130
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


Created version '30' of model 'AdultIncome_LogisticRegression'.


## Модели

In [24]:
best_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
        ("num", StandardScaler(), num_features),
    ]
)

models = {
    "LogReg": LogisticRegression(C=1.0, solver="newton-cg", max_iter=100, random_state=RANDOM_STATE),
    "DecisionTree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
}

for model_name, model in models.items():
    run_name = f"model_{model_name}"
    hypothesis = f"Сравнение моделей. {model_name}"
    
    run_experiment(
        run_name=run_name,
        hypothesis=hypothesis,
        model=model,
        preprocessor=best_preprocessor,
        X_train_exp=X_train,
        y_train_exp=y_train,
        X_test_exp=X_test,
        y_test_exp=y_test,
        cat_feat=cat_features,
        num_feat=num_features
    )

/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:48:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[model_LogReg] ROC-AUC: 0.9014
🏃 View run model_LogReg at: http://158.160.242.172:5000/#/experiments/35/runs/d99943de54a645e8a08db16531af00f9
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:48:24 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successful

[model_DecisionTree] ROC-AUC: 0.8904
🏃 View run model_DecisionTree at: http://158.160.242.172:5000/#/experiments/35/runs/c1069324369a4024a4d1675c1c405281
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:48:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[model_RandomForest] ROC-AUC: 0.9109
🏃 View run model_RandomForest at: http://158.160.242.172:5000/#/experiments/35/runs/3d052a49a6f247a490d8d67609f4f56d
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


Created version '25' of model 'AdultIncome_RandomForestClassifier'.


## Тюним лучшую модель

In [27]:
best_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
        ("num", StandardScaler(), num_features),
    ]
)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

keys = param_grid.keys()
combinations = itertools.product(*param_grid.values())

for combo in combinations:
    params = dict(zip(keys, combo))
    params["random_state"] = RANDOM_STATE
    
    run_name = f"RF_est{params['n_estimators']}_md{params['max_depth']}_mss{params['min_samples_split']}"
    hypothesis = f"Тюнинг RF: n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, min_samples_split={params['min_samples_split']}"
    
    model = RandomForestClassifier(**params)
    
    run_experiment(
        run_name=run_name,
        hypothesis=hypothesis,
        model=model,
        preprocessor=best_preprocessor,
        X_train_exp=X_train,
        y_train_exp=y_train,
        X_test_exp=X_test,
        y_test_exp=y_test,
        cat_feat=cat_features,
        num_feat=num_features
    )

/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:55:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[RF_est50_md10_mss2] ROC-AUC: 0.9101
🏃 View run RF_est50_md10_mss2 at: http://158.160.242.172:5000/#/experiments/35/runs/ff00665a33914ed495a4dca605636678
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:55:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[RF_est50_md10_mss5] ROC-AUC: 0.9110
🏃 View run RF_est50_md10_mss5 at: http://158.160.242.172:5000/#/experiments/35/runs/bb7e71832f284fa79bd8a480e4b7a435
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:55:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

...

[RF_est200_md20_mss5] ROC-AUC: 0.9138
🏃 View run RF_est200_md20_mss5 at: http://158.160.242.172:5000/#/experiments/35/runs/d875acf3008b4917b4ff88f5f25c0991
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:57:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[RF_est200_mdNone_mss2] ROC-AUC: 0.8901
🏃 View run RF_est200_mdNone_mss2 at: http://158.160.242.172:5000/#/experiments/35/runs/84b09660c29e47028862541f862caf22
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 18:58:09 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[RF_est200_mdNone_mss5] ROC-AUC: 0.9016
🏃 View run RF_est200_mdNone_mss5 at: http://158.160.242.172:5000/#/experiments/35/runs/e4d720d0b7f34806a7d8123377588289
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


## Зависимость от размера данных

In [29]:
fractions = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]

for frac in fractions:
    n_samples = int(len(X_train) * frac)
    
    X_train_sub = X_train.iloc[:n_samples]
    y_train_sub = y_train.iloc[:n_samples]
    
    run_name = f"DataSize_{int(frac * 100)}pct"
    hypothesis = f"Влияние размера данных: {int(frac * 100)}% ({n_samples} строк)"
    
    best_model = RandomForestClassifier(
        n_estimators=200, 
        max_depth=20, 
        min_samples_split=5, 
        random_state=RANDOM_STATE
    )
    
    run_experiment(
        run_name=run_name,
        hypothesis=hypothesis,
        model=best_model,
        preprocessor=best_preprocessor,
        X_train_exp=X_train_sub,
        y_train_exp=y_train_sub,
        X_test_exp=X_test,
        y_test_exp=y_test,
        cat_feat=cat_features,
        num_feat=num_features
    )

/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 20:29:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[DataSize_1pct] ROC-AUC: 0.8836
🏃 View run DataSize_1pct at: http://158.160.242.172:5000/#/experiments/35/runs/d628526085f347a9b484b7750f08508d
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 20:29:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[DataSize_5pct] ROC-AUC: 0.8939
🏃 View run DataSize_5pct at: http://158.160.242.172:5000/#/experiments/35/runs/35ec63a0f58246aab14ebb75087bea12
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 20:30:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

...

[DataSize_25pct] ROC-AUC: 0.9050
🏃 View run DataSize_25pct at: http://158.160.242.172:5000/#/experiments/35/runs/ffd1660d6749430fb01568c6e1dcbc15
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 20:30:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[DataSize_50pct] ROC-AUC: 0.9101
🏃 View run DataSize_50pct at: http://158.160.242.172:5000/#/experiments/35/runs/6e45fedc0ec84df6985e1e874945bbbd
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


/home/danii/projects/3_education/avito_academy/2_ML_System_Design/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/08 20:30:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered

[DataSize_100pct] ROC-AUC: 0.9138
🏃 View run DataSize_100pct at: http://158.160.242.172:5000/#/experiments/35/runs/5e30bb7ba38d44428b469cc2a34b05d0
🧪 View experiment at: http://158.160.242.172:5000/#/experiments/35


# Проверка загрузки лучшей модели

In [31]:
import mlflow
import numpy as np
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, average_precision_score
)

model_uri = "mlflow-artifacts:/35/d875acf3008b4917b4ff88f5f25c0991/artifacts/model"

loaded_model = mlflow.sklearn.load_model(model_uri)

y_proba_loaded = loaded_model.predict_proba(X_test)[:, 1]
y_pred_loaded = np.where(y_proba_loaded >= 0.5, 1, 0)

loaded_metrics = {
    "ROC-AUC": roc_auc_score(y_test, y_proba_loaded),
    "PR-AUC": average_precision_score(y_test, y_proba_loaded),
    "Accuracy": accuracy_score(y_test, y_pred_loaded),
    "Precision": precision_score(y_test, y_pred_loaded),
    "Recall": recall_score(y_test, y_pred_loaded),
    "F1-score": f1_score(y_test, y_pred_loaded),
}

for metric_name, value in loaded_metrics.items():
    print(f"{metric_name}: {value:.4f}")

ROC-AUC: 0.9138
PR-AUC: 0.7901
Accuracy: 0.8597
Precision: 0.7687
Recall: 0.5923
F1-score: 0.6691


# Выгрузка финальной таблицы со всеми запусками

In [35]:
EXPERIMENT_NAME = "homework-dvsobolev.ext" 
runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])

if "metrics.roc_auc" in runs_df.columns:
    runs_df = runs_df.sort_values(by="metrics.roc_auc", ascending=False)

columns_to_show = [
    "tags.mlflow.runName", 
    "metrics.roc_auc", 
    "metrics.accuracy",
    "tags.mlflow.note.content"
]

existing_columns = [col for col in columns_to_show if col in runs_df.columns]
clean_df = runs_df[existing_columns]
clean_df

,tags.mlflow.runName,metrics.roc_auc,metrics.accuracy,tags.mlflow.note.content
0,DataSize_100pct,0.913836,0.859658,Влияние размера данных: 100% (22792 строк)
8,RF_est200_md20_mss5,0.913836,0.859658,"Тюнинг RF: n_estimators=200, max_depth=20, min..."
14,RF_est100_md20_mss5,0.913662,0.859249,"Тюнинг RF: n_estimators=100, max_depth=20, min..."
20,RF_est50_md20_mss5,0.912406,0.858020,"Тюнинг RF: n_estimators=50, max_depth=20, min_..."
9,RF_est200_md20_mss2,0.911566,0.857406,"Тюнинг RF: n_estimators=200, max_depth=20, min..."
10,RF_est200_md10_mss5,0.911432,0.852185,"Тюнинг RF: n_estimators=200, max_depth=10, min..."
11,RF_est200_md10_mss2,0.911281,0.853004,"Тюнинг RF: n_estimators=200, max_depth=10, min..."
16,RF_est100_md10_mss5,0.911060,0.852697,"Тюнинг RF: n_estimators=100, max_depth=10, min..."
22,RF_est50_md10_mss5,0.911013,0.851162,"Тюнинг RF: n_estimators=50, max_depth=10, min_..."
17,RF_est100_md10_mss2,0.910892,0.853209,"Тюнинг RF: n_estimators=100, max_depth=10, min..."
